**<h3>LEVEL 1 - TASK 3 - DATA CLEANING</h3>**

In [1]:
# Importing necessary libraries
import pandas as pd
import numpy as np

# Creating a DataFrame
data = pd.read_csv('bank_churn_dirty.csv')  
data.head()



,customer_id,nome,email,numero_conta,data_nascimento,cidade,country,gender,age,credit_score,tenure,balance,limite_credito,products_number,credit_card,active_member,estimated_salary,churn
0,15631460.0,Camila Martins Lima,camila.martins@hotmail.com,83406-0,04/10/1982,Barcelona,Spain,Female,42,671,3,0.00,0.00,2,1,1,128449.33,0
1,15700300.0,Patrícia Rocha Alves,patricia.rocha@uol.com.br,83680-0,18/12/1980,Berlin,Germany,Female,44,674,4,131593.85,202324.81,1,0,1,171345.02,1
2,NaN,ANDRÉ COSTA ARAÚJO,andre.costa@gmail.com,73884-8,1985-08-26,Berlin,Germany,Male,39,789,7,124828.46,356669.15,2,1,1,124411.08,0
3,15760456.0,Silvana Costa Ferreira,silvana.costa@outlook.com,43299-9,1986-04-04,Paris,France,Female,38,731,10,123711.73,262028.34,2,1,0,171340.68,1
4,15638730.0,Natália Santos Martins,natalia.santos@yahoo.com.br,56717-2,2003-04-11,Paris,France,Female,-1,711,0,82844.33,207795.24,2,0,1,1408.68,0


**<h3>Step 1: Data Quality Assessment</h3>**

In [2]:
# Basic inspection
print("Dataset Shape:", data.shape)

print("\nColumn Data Types:")
display(data.dtypes)

print("\nMissing Values:")
display(data.isnull().sum())

print("\nDuplicate Rows:", data.duplicated().sum())

Dataset Shape: (10150, 18)

Column Data Types:


customer_id         float64
nome                    str
email                   str
numero_conta            str
data_nascimento         str
cidade                  str
country                 str
gender                  str
age                   int64
credit_score          int64
tenure                int64
balance             float64
limite_credito      float64
products_number       int64
credit_card           int64
active_member         int64
estimated_salary    float64
churn                 int64
dtype: object


Missing Values:


customer_id         203
nome                  0
email               405
numero_conta          0
data_nascimento       0
cidade                0
country               0
gender                0
age                   0
credit_score          0
tenure                0
balance               0
limite_credito        0
products_number       0
credit_card           0
active_member         0
estimated_salary    605
churn                 0
dtype: int64


Duplicate Rows: 149


In [3]:
# Data Quality Report

quality_report = pd.DataFrame({
    "Column": data.columns,
    "Data Type": data.dtypes.astype(str).values,
    "Null Count": data.isnull().sum().values,
    "Null %": (data.isnull().mean() * 100).round(2).values,
    "Unique Values": data.nunique().values
})

display(quality_report)

,Column,Data Type,Null Count,Null %,Unique Values
0,customer_id,float64,203,2.00,9798
1,nome,str,0,0.00,80
2,email,str,405,3.99,482
3,numero_conta,str,0,0.00,9948
4,data_nascimento,str,0,0.00,8018
5,cidade,str,0,0.00,38
6,country,str,0,0.00,3
7,gender,str,0,0.00,2
8,age,int64,0,0.00,71
9,credit_score,int64,0,0.00,461


*Numeric range/anomaly report*

In [4]:
numeric_columns = data.select_dtypes(include=np.number).columns

range_report = pd.DataFrame({
    "Column": numeric_columns,
    "Min": [data[col].min() for col in numeric_columns],
    "Max": [data[col].max() for col in numeric_columns],
    "Mean": [data[col].mean() for col in numeric_columns],
    "Median": [data[col].median() for col in numeric_columns]
})

display(range_report)

,Column,Min,Max,Mean,Median
0,customer_id,15565701.00,15815690.00,1.569098e+07,1.569059e+07
1,age,-1.00,92.00,3.768995e+01,3.700000e+01
2,credit_score,350.00,999.00,6.611561e+02,6.550000e+02
3,tenure,0.00,10.00,5.011626e+00,5.000000e+00
4,balance,-204510.94,999999.99,8.765049e+04,9.475377e+04
5,limite_credito,0.00,609777.08,1.322732e+05,1.137457e+05
6,products_number,1.00,4.00,1.528473e+00,1.000000e+00
7,credit_card,0.00,1.00,7.058128e-01,1.000000e+00
8,active_member,0.00,1.00,5.145813e-01,1.000000e+00
9,estimated_salary,11.58,199992.48,1.000682e+05,1.001534e+05


In [5]:
# Business-rule anomaly checks

anomalies = {
    "Invalid Age (<18 or >100)": ((data["age"] < 18) | (data["age"] > 100)).sum(),
    "Invalid Credit Score (<300 or >850)": (
        (data["credit_score"] < 300) | (data["credit_score"] > 850)
    ).sum(),
    "Negative Balance": (data["balance"] < 0).sum(),
    "Invalid Products Number": (
        (data["products_number"] < 1) | (data["products_number"] > 4)
    ).sum(),
    "Invalid Credit Card Flag": (~data["credit_card"].isin([0, 1])).sum(),
    "Invalid Active Member Flag": (~data["active_member"].isin([0, 1])).sum(),
    "Invalid Churn Flag": (~data["churn"].isin([0, 1])).sum()
}

anomaly_report = pd.DataFrame(
    list(anomalies.items()),
    columns=["Anomaly", "Count"]
)

display(anomaly_report)

,Anomaly,Count
0,Invalid Age (<18 or >100),304
1,Invalid Credit Score (<300 or >850),303
2,Negative Balance,316
3,Invalid Products Number,0
4,Invalid Credit Card Flag,0
5,Invalid Active Member Flag,0
6,Invalid Churn Flag,0


**Data Quality Observations**

The raw dataset was first inspected to establish a baseline for data quality. The dataset contains 10,150 rows and 18 columns. The initial assessment identified missing values in customer_id, email, and estimated_salary, as well as 149 duplicate rows. Several data type issues were also identified, including identifiers stored as numeric values and dates stored as text. Business-rule checks revealed potentially invalid values in variables such as age, credit score, and account balance.

These issues were identified before any cleaning was performed to ensure that each problem could be addressed using an appropriate strategy. Statistical imputation will only be applied to suitable numerical variables, while identifiers and contact information will not be artificially generated. Duplicate records will be removed, inconsistent categorical formatting will be standardized, invalid values will be investigated, and data types will be corrected. The original results from this assessment will serve as the before-cleaning baseline for the final before-versus-after comparison.

**<h3>Step 2: Inspect missing values</h3>**

*A. Missing Data Handling*

Missing values were reviewed column by column rather than applying one method to the entire dataset. The treatment depends on whether the column is an identifier, categorical variable, or numerical variable, and on how important the variable is for the analysis.

In [6]:
# Detailed missing-value analysis

missing_report = pd.DataFrame({
    "Missing Count": data.isnull().sum(),
    "Missing Percentage": (data.isnull().mean() * 100).round(2)
})

missing_report = missing_report[missing_report["Missing Count"] > 0]

display(missing_report)

,Missing Count,Missing Percentage
customer_id,203,2.00
email,405,3.99
estimated_salary,605,5.96


*B. Estimated Salary Treatment*

The estimated_salary column contains missing values. Because salary is a numerical variable, the missing values will be replaced with the median salary. Median imputation was selected because financial variables can contain extreme values, and the median is less sensitive to outliers than the mean. This approach preserves the observations instead of deleting otherwise useful customer records.

In [7]:
# Store the median before imputation for documentation
salary_median = data["estimated_salary"].median()

print("Median estimated salary:", salary_median)

# Impute missing salaries
data["estimated_salary"] = data["estimated_salary"].fillna(salary_median)

print(
    "Remaining missing estimated_salary values:",
    data["estimated_salary"].isnull().sum()
)

Median estimated salary: 100153.43
Remaining missing estimated_salary values: 0


*C. Email Treatment*

Missing email values will be retained as missing rather than artificially generated. Email addresses are identifiers/contact information and cannot be reliably inferred from the available variables. Since email is not required for the planned churn analysis, retaining the missing values avoids introducing fabricated information.

*D. Customer ID Treatment*

customer_id is an identifier rather than an analytical measurement. The 203 missing values will not be replaced with statistical values such as the mean or median. Creating artificial customer IDs could incorrectly imply that the records belong to identifiable customers. The missing IDs will therefore be retained as missing unless the business provides a reliable source for recovering them.

In [8]:
# Verify missing values after the first missing-data treatment

missing_after_salary = pd.DataFrame({
    "Missing Count": data.isnull().sum(),
    "Missing Percentage": (data.isnull().mean() * 100).round(2)
})

display(
    missing_after_salary[
        missing_after_salary["Missing Count"] > 0
    ]
)

,Missing Count,Missing Percentage
customer_id,203,2.00
email,405,3.99


*Step 2 Decisons Documentation*

In [9]:
missing_decisions = pd.DataFrame({
    "Column": [
        "customer_id",
        "email",
        "estimated_salary"
    ],
    "Missing Count Before": [
        203,
        405,
        605
    ],
    "Strategy": [
        "Retain as missing",
        "Retain as missing",
        "Median imputation"
    ],
    "Reason": [
        "Identifier; statistical imputation would create a false identity",
        "Cannot reliably infer email addresses; not required for core analysis",
        "Numerical financial variable; median is less sensitive to extreme values"
    ]
})

display(missing_decisions)

,Column,Missing Count Before,Strategy,Reason
0,customer_id,203,Retain as missing,Identifier; statistical imputation would creat...
1,email,405,Retain as missing,Cannot reliably infer email addresses; not req...
2,estimated_salary,605,Median imputation,Numerical financial variable; median is less s...


**<h3>STEP 3: Duplicate Removal</h3>**

*3.1 Identifying Duplicate Records*

In [10]:
# inspecting the duplicates

duplicates = data[data.duplicated(keep=False)]

display(duplicates.head(25))

,customer_id,nome,email,numero_conta,data_nascimento,cidade,country,gender,age,credit_score,tenure,balance,limite_credito,products_number,credit_card,active_member,estimated_salary,churn
8,15689061.0,Marcos Pereira Rocha,marcos.pereira@terra.com.br,68547-4,23/06/1956,Marseille,France,Male,68,611,5,82547.11,229393.65,2,1,1,146448.01,0
17,15621466.0,Mateus Carvalho Alves,mateus.carvalho@uol.com.br,71355-9,23/07/1986,Hamburg,Germany,Male,38,606,3,99897.53,195156.04,1,0,0,37054.65,0
62,15634281.0,Débora Ribeiro Lima,debora.ribeiro@terra.com.br,60702-3,15/02/1981,Stuttgart,Germany,Female,43,720,10,110822.90,153329.41,1,0,0,72861.94,0
132,15728963.0,Camila Martins Lima,NaN,00358-2,07/11/1973,cologne,Germany,Female,51,617,10,167273.71,437651.08,1,0,0,93439.75,1
159,15568506.0,Juliana Alves Pereira,juliana.alves@yahoo.com.br,29661-1,25/05/1993,Munich,Germany,Female,31,524,10,67238.98,136733.02,2,1,1,161811.23,0
198,15636572.0,Silvana Costa Ferreira,silvana.costa@hotmail.com,65633-8,14/05/1992,nantes,France,Female,32,760,7,0.00,0.00,2,1,1,105969.05,0
256,15645200.0,Priscila Rocha Nunes,priscila.rocha@hotmail.com,22560-3,1970-09-09,Cologne,Germany,Female,54,581,2,152508.99,397359.78,1,1,0,187597.98,1
309,15804864.0,Priscila Rocha Nunes,priscila.rocha@hotmail.com,84181-1,22/02/1997,Nice,France,Female,27,999,5,79336.61,85252.35,1,1,1,26170.08,0
416,15682995.0,Renata Souza Ribeiro,renata.souza@terra.com,22032-2,1992-03-25,Toulouse,France,Female,32,600,1,78535.25,111284.12,1,1,0,64349.60,0
437,15598782.0,Mariana Costa Santos,mariana.costa@uol.com.br,92829-5,1994-10-15,Berlin,Germany,Female,30,755,6,154221.37,203199.44,2,0,1,62688.55,0


In [11]:
# confirming the exact duplication
print("Duplicate rows:", data.duplicated().sum())
print("Total rows before removal:", len(data))

Duplicate rows: 149
Total rows before removal: 10150


In [12]:
# Remove exact duplicate rows

df = data.drop_duplicates()

print("Rows after duplicate removal:", len(df))

Rows after duplicate removal: 10001


In [13]:
# Verify duplicate removal

remaining_duplicates = df.duplicated().sum()

print("Remaining duplicate rows:", remaining_duplicates)

Remaining duplicate rows: 0


In [14]:
# duplicate summary report
duplicate_summary = pd.DataFrame({
    "Metric": [
        "Rows",
        "Duplicate Rows"
    ],
    "Before": [
        10150,
        149
    ],
    "After": [
        len(df),
        df.duplicated().sum()
    ]
})

display(duplicate_summary)

,Metric,Before,After
0,Rows,10150,10001
1,Duplicate Rows,149,0


**Duplicate Removal — Observation**

The initial data-quality assessment identified 149 exact duplicate rows in the dataset. The duplicate records were inspected before removal to confirm that they represented repeated observations rather than legitimate records with shared customer information. Since the duplicated rows contained identical values across all columns, they were considered redundant and removed using drop_duplicates().

The dataset decreased from 10,150 rows to 10,001 rows, meaning 149 duplicate records were removed. A verification check confirmed that no exact duplicate rows remain.

**<h3>Step 4:  Standardisation</h3 >**

*Standardising the names*

In [15]:
# how many names are ALL CAPS before the fix
print("ALL CAPS names before:", df["nome"].str.isupper().sum())

# strip() removes stray spaces at the ends, title() capitalises the first letter of each word
df["nome"] = df["nome"].str.strip().str.title()

# confirm nothing is ALL CAPS any more
print("ALL CAPS names after:", df["nome"].str.isupper().sum())
df["nome"].head(10)

ALL CAPS names before: 800
ALL CAPS names after: 0


0       Camila Martins Lima
1      Patrícia Rocha Alves
2        André Costa Araújo
3    Silvana Costa Ferreira
4    Natália Santos Martins
5        André Costa Araújo
6      Carlos Eduardo Silva
7     Larissa Pereira Gomes
8      Marcos Pereira Rocha
9      Mariana Costa Santos
Name: nome, dtype: str

**Name Standardisation — Observation**

800 names were written in ALL CAPS. They were converted to normal capitalisation so that the same person is not written two different ways, which matters for grouping and matching. No names remain in ALL CAPS.

*Standardising the age*

In [58]:
#Standardising gender

df["gender"].value_counts()

gender
Male      5458
Female    4543
Name: count, dtype: int64

*Standardaising Cidade/city*

In [52]:
#Standardising cidade

df['cidade'].unique()


<ArrowStringArray>
['Barcelona',    'Berlin',     'Paris',    'Munich',    'Madrid',    'Nantes',
 'Marseille',      'Nice',      'Lyon',   'Cologne', 'Frankfurt',  'Bordeaux',
   'Sevilla',   'Hamburg',  'Toulouse',    'Bilbao',  'Valencia', 'Stuttgart',
  'Zaragoza']
Length: 19, dtype: str

In [57]:
df["cidade"].value_counts()

cidade
Lyon         763
Bordeaux     736
Toulouse     726
Nantes       712
Paris        711
Nice         687
Marseille    680
Barcelona    449
Berlin       435
Cologne      432
Frankfurt    422
Zaragoza     419
Madrid       418
Stuttgart    418
Sevilla      406
Munich       405
Bilbao       398
Hamburg      397
Valencia     387
Name: count, dtype: int64

In [19]:
#standardise capitalization
df["cidade"] = df["cidade"].str.strip().str.title()

#confirming the capitalization
df["cidade"].value_counts(dropna=False)

cidade
Lyon         763
Bordeaux     736
Toulouse     726
Nantes       712
Paris        711
Nice         687
Marseille    680
Barcelona    449
Berlin       435
Cologne      432
Frankfurt    422
Zaragoza     419
Madrid       418
Stuttgart    418
Sevilla      406
Munich       405
Bilbao       398
Hamburg      397
Valencia     387
Name: count, dtype: int64

*Standardising Dates*

In [20]:
# Inspect the original date values before cleaning

print(df["data_nascimento"].head(20))
print("\nOriginalData type:", df["data_nascimento"].dtype)
print("\nOriginal missing values:", df["data_nascimento"].isna().sum())

0     04/10/1982
1     18/12/1980
2     1985-08-26
3     1986-04-04
4     2003-04-11
5     1996-03-07
6     20/08/1990
7     02/10/1978
8     23/06/1956
9     1967-01-06
10    07/11/1979
11    21/11/1981
12    1959-11-24
13    02/05/1964
14    1985-08-28
15    1973-05-20
16    1949-02-23
17    23/07/1986
18    1963-05-05
19    1988-01-01
Name: data_nascimento, dtype: str

OriginalData type: str

Original missing values: 0


In [21]:
# 2. Convert to string for format identification
dates = df["data_nascimento"].astype("string")

# Identify dates containing "/"
slash_dates = dates.str.contains("/", na=False)

# Identify dates containing "-"
hyphen_dates = dates.str.contains("-", na=False)

print("Slash-format dates:", slash_dates.sum())
print("Hyphen-format dates:", hyphen_dates.sum())

Slash-format dates: 5996
Hyphen-format dates: 4005


In [22]:
# parsing the slash-format dates using the "dd/mm/yyyy" format
parsed_slash = pd.to_datetime(
    dates[slash_dates],
    format="%d/%m/%Y",
    errors="coerce"
)


# parsing the hyphen-format dates using the "yyyy-mm-dd" format
parsed_hyphen = pd.to_datetime(
    dates[hyphen_dates],
    format="%Y-%m-%d",
    errors="coerce"
)

In [23]:
#  Create a clean datetime Series
clean_dates = pd.Series(
    pd.NaT,
    index=df.index,
    dtype="datetime64[ns]"
)

# Combine the parsed dates into the clean_dates Series
clean_dates.loc[slash_dates] = parsed_slash
clean_dates.loc[hyphen_dates] = parsed_hyphen

# Replace the original column
df["data_nascimento"] = clean_dates

# confirming the number of missing/invalid dates after the cleaning process
print(
    "Remaining missing/invalid dates:",
    df["data_nascimento"].isna().sum()
)

print("\nAfter standardisation:")
print("Data type:", df["data_nascimento"].dtype)
print("Missing/invalid dates:", df["data_nascimento"].isna().sum())

display(df["data_nascimento"].head(20))


Remaining missing/invalid dates: 0

After standardisation:
Data type: datetime64[ns]
Missing/invalid dates: 0


0    1982-10-04
1    1980-12-18
2    1985-08-26
3    1986-04-04
4    2003-04-11
5    1996-03-07
6    1990-08-20
7    1978-10-02
8    1956-06-23
9    1967-01-06
10   1979-11-07
11   1981-11-21
12   1959-11-24
13   1964-05-02
14   1985-08-28
15   1973-05-20
16   1949-02-23
17   1986-07-23
18   1963-05-05
19   1988-01-01
Name: data_nascimento, dtype: datetime64[ns]

**Date Standardisation — Observation**

The data_nascimento column was inspected before conversion because the dataset contained dates in different formats. The inspection showed two valid formats: DD/MM/YYYY and YYYY-MM-DD. A total of 6,096 records used the DD/MM/YYYY format, while 4,054 records used the YYYY-MM-DD format.

Instead of applying a single automatic date conversion to the entire column, each format was parsed separately using its appropriate format specification. The converted values were then combined into a single datetime column.

After standardisation, the data_nascimento column was successfully converted from string to datetime64[ns]. All 10,150 records were successfully parsed, leaving 0 missing or invalid dates.

This approach was chosen because inspecting the original values first prevented valid dates from being incorrectly classified as missing due to inconsistent formatting.

# Step 5 — Outlier Detection

In [24]:
# inspecting the numeric columns for outliers using the IQR method

numeric_columns = df.select_dtypes(include="number").columns

print(numeric_columns)

Index(['customer_id', 'age', 'credit_score', 'tenure', 'balance',
       'limite_credito', 'products_number', 'credit_card', 'active_member',
       'estimated_salary', 'churn'],
      dtype='str')


In [25]:
df[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
customer_id,9798.0,1.569096e+07,71996.551700,15565701.00,15628344.25,15690633.50,15753549.75,15815690.00
age,10001.0,3.771063e+01,12.380091,-1.00,31.00,37.00,44.00,92.00
credit_score,10001.0,6.612080e+02,112.195345,350.00,586.00,655.00,726.00,999.00
tenure,10001.0,5.012699e+00,2.892047,0.00,3.00,5.00,7.00,10.00
balance,10001.0,8.752527e+04,147918.872898,-204510.94,0.00,94753.55,128015.83,999999.99
limite_credito,10001.0,1.320730e+05,128735.763598,0.00,0.00,113481.20,227312.62,609777.08
products_number,10001.0,1.530147e+00,0.581649,1.00,1.00,1.00,2.00,4.00
credit_card,10001.0,7.055294e-01,0.455827,0.00,0.00,1.00,1.00,1.00
active_member,10001.0,5.150485e-01,0.499798,0.00,0.00,1.00,1.00,1.00
estimated_salary,10001.0,1.000765e+05,55789.076220,11.58,54144.01,100153.43,146111.35,199992.48


In [26]:
# creating a function to find outliers using the IQR method
def find_iqr_outliers(dataframe, column):
    
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = dataframe[
        (dataframe[column] < lower_bound) |
        (dataframe[column] > upper_bound)
    ]
    
    print(f"Column: {column}")
    print(f"Q1: {Q1}")
    print(f"Q3: {Q3}")
    print(f"IQR: {IQR}")
    print(f"Lower bound: {lower_bound}")
    print(f"Upper bound: {upper_bound}")
    print(f"Potential outliers: {len(outliers)}")
    
    return outliers

In [27]:
# Find outliers for each numerical column
print ("\n Age outliers:")
age_outliers = find_iqr_outliers(df, "age")

print ("\n Credit Score outliers:")
credit_score_outliers = find_iqr_outliers(df, "credit_score")

print ("\n Balance outliers:")
balance_outliers = find_iqr_outliers(df, "balance")


 Age outliers:
Column: age
Q1: 31.0
Q3: 44.0
IQR: 13.0
Lower bound: 11.5
Upper bound: 63.5
Potential outliers: 610

 Credit Score outliers:
Column: credit_score
Q1: 586.0
Q3: 726.0
IQR: 140.0
Lower bound: 376.0
Upper bound: 936.0
Potential outliers: 312

 Balance outliers:
Column: balance
Q1: 0.0
Q3: 128015.83
IQR: 128015.83
Lower bound: -192023.745
Upper bound: 320039.575
Potential outliers: 203


*Step 5A — Investigate Age  column*

In [28]:
# Checking the range of ages in the dataset
print("Minimum age:", df["age"].min())
print("Maximum age:", df["age"].max())

# Count clearly invalid ages
invalid_age = df[
    (df["age"] < 18) |
    (df["age"] > 100)
]

print("Invalid ages:", len(invalid_age))

display(
    invalid_age[["customer_id", "age", "data_nascimento"]].sort_values("age").head(20)
)

Minimum age: -1
Maximum age: 92
Invalid ages: 300


,customer_id,age,data_nascimento
4,15638730.0,-1,2003-04-11
10,15566494.0,-1,1979-11-07
21,15720919.0,-1,1982-04-23
61,15790455.0,-1,1974-01-19
63,NaN,-1,1994-11-18
76,15627365.0,-1,1978-04-27
144,15808326.0,-1,1990-01-23
155,15760550.0,-1,1985-08-12
175,15654292.0,-1,1991-07-15
204,15769312.0,-1,1976-01-28


In [29]:
# Reference date used by the dataset
reference_date = pd.Timestamp("2024-12-28")
dob = df["data_nascimento"]


birthday_not_yet = (
    (reference_date.month < dob.dt.month)
    | ((reference_date.month == dob.dt.month) & (reference_date.day < dob.dt.day))
)
calculated_age = reference_date.year - dob.dt.year - birthday_not_yet.astype(int)

# check the method: on rows where age was already valid, how often does it agree?
valid_age = df["age"] != -1
match_rate = (calculated_age[valid_age] == df.loc[valid_age, "age"]).mean()
print(f"Method agrees with existing valid ages: {match_rate:.1%}")




Method agrees with existing valid ages: 100.0%


In [30]:
# Calculate age from date of birth
calculated_age = (
    reference_date.year - df["data_nascimento"].dt.year
    - (
        (df["data_nascimento"].dt.month > reference_date.month)
        |
        (
            (df["data_nascimento"].dt.month == reference_date.month)
            & (df["data_nascimento"].dt.day > reference_date.day)
        )
    )
)

# Check the calculated ages for negative-age records
display(
    df.loc[
        df["age"] < 0,
        ["customer_id", "age", "data_nascimento"]
    ].assign(
        calculated_age=calculated_age[df["age"] < 0]
    ).head(20)
)

,customer_id,age,data_nascimento,calculated_age
4,15638730.0,-1,2003-04-11,21
10,15566494.0,-1,1979-11-07,45
21,15720919.0,-1,1982-04-23,42
61,15790455.0,-1,1974-01-19,50
63,NaN,-1,1994-11-18,30
76,15627365.0,-1,1978-04-27,46
144,15808326.0,-1,1990-01-23,34
155,15760550.0,-1,1985-08-12,39
175,15654292.0,-1,1991-07-15,33
204,15769312.0,-1,1976-01-28,48


In [31]:
#replacing negative ages with calculated ages
negative_age_mask = df["age"] < 0

df.loc[negative_age_mask, "age"] = calculated_age[negative_age_mask]

#verify that all negative ages have been replaced
print("Negative ages remaining:", (df["age"] < 0).sum())

print("Minimum age:", df["age"].min())
print("Maximum age:", df["age"].max())

Negative ages remaining: 0
Minimum age: 18
Maximum age: 92


**Age Outlier Handling**

The age column contained invalid negative values, represented by -1. These values were not treated as legitimate statistical outliers because a negative age is not meaningful for a customer record. Since the dataset contained a valid data_nascimento value for these records, the correct age was calculated from the date of birth using the dataset's reference date of 28 December 2024. The calculated ages were then used to replace the invalid negative values. This approach was preferred over mean or median imputation because the date of birth provided a more accurate basis for recovering the original age.

*Step 5B — Investigate Credit Score*

In [32]:
#checking the range of credit scores in the dataset
print("Minimum credit score:", df["credit_score"].min())
print("Maximum credit score:", df["credit_score"].max())

# Count clearly invalid credit scores
invalid_credit_score = df[
    (df["credit_score"] < 300) |
    (df["credit_score"] > 850)
]

print("Invalid credit scores:", len(invalid_credit_score))

display(
    invalid_credit_score[
        ["customer_id",
                    "credit_score",
                    "age",
                    "balance",
                    "products_number",
                    "credit_card",
                    "active_member",
                    "churn"
        ]
    ].head(20)
)




Minimum credit score: 350
Maximum credit score: 999
Invalid credit scores: 300


,customer_id,credit_score,age,balance,products_number,credit_card,active_member,churn
68,15760641.0,999,26,106648.98,1,0,1,0
71,15584452.0,999,41,0.00,2,0,0,0
145,15604813.0,999,40,-0.00,2,0,1,0
186,15781326.0,999,35,95478.17,1,0,0,0
189,15705403.0,999,46,106521.49,1,0,1,0
307,15577695.0,999,41,148088.11,1,1,0,0
309,15804864.0,999,27,79336.61,1,1,1,0
388,15586674.0,999,58,216109.88,1,0,1,1
448,15617518.0,999,36,-89409.95,1,1,1,0
486,15612525.0,999,57,0.00,1,0,0,1


In [33]:
# Replace the invalid credit score value of 999 with NaN
df["credit_score"] = df["credit_score"].replace(999, np.nan)

# Check how many are now missing
print("Missing credit scores:", df["credit_score"].isna().sum())

Missing credit scores: 300


In [34]:
# calculate the median credit score for imputation
credit_score_median = df["credit_score"].median()

print("Median credit score:", credit_score_median)

# Impute missing credit scores with the median
df["credit_score"] = df["credit_score"].fillna(credit_score_median)

# verify that there are no more missing credit scores
print("Missing credit scores after cleaning:",
      df["credit_score"].isna().sum())

print("Minimum credit score:",
      df["credit_score"].min())

print("Maximum credit score:",
      df["credit_score"].max())

Median credit score: 652.0
Missing credit scores after cleaning: 0
Minimum credit score: 350.0
Maximum credit score: 850.0


In [35]:
print(
    df["credit_score"].describe()
)

print(
    "Values above 850:",
    (df["credit_score"] > 850).sum()
)

count    10001.000000
mean       650.799020
std         95.178139
min        350.000000
25%        586.000000
50%        652.000000
75%        715.000000
max        850.000000
Name: credit_score, dtype: float64
Values above 850: 0


**Credit Score Outlier Handling**

The credit score column contained 300 records with a value of 999. The dataset documentation indicates that noise was intentionally injected into the dataset. The value 999 was therefore treated as an injected erroneous value rather than a genuine credit score.

Since the original credit scores for these records could not be determined from the available data, the 999 values were first converted to missing values (NaN). The missing values were then replaced using the median credit score.

Median imputation was selected because it is less sensitive to extreme values than mean imputation and helps preserve the affected customer records.

Other values identified as IQR outliers were retained because being statistically unusual does not necessarily mean that the values are erroneous.

*Step 5C — Investigate Balance* 

In [36]:
# Identify customers with negative account balances
negative_balance = df[df["balance"] < 0]

print("Number of negative balances:", len(negative_balance))

display(
    negative_balance[
        ["customer_id", "balance", "age", "credit_score"]
    ].head(20)
)

# Calculate statistics for negative balances
print("Most negative balance:", negative_balance["balance"].min())
print("Average negative balance:", negative_balance["balance"].mean())
print("Number of negative balances:", len(negative_balance))

Number of negative balances: 308


,customer_id,balance,age,credit_score
29,15657468.0,-123805.03,53,711.0
36,15727696.0,-147249.29,42,592.0
50,15798024.0,-92242.34,84,537.0
79,15661670.0,-107818.63,31,524.0
97,15631838.0,-108166.09,61,606.0
110,15746258.0,-101486.96,29,622.0
140,15694717.0,-79731.91,37,544.0
258,15774250.0,-159024.71,42,532.0
291,15763111.0,-124577.15,67,808.0
310,15724282.0,-164113.04,44,540.0


Most negative balance: -204510.94
Average negative balance: -120232.18045454545
Number of negative balances: 308


In [37]:
#Inspect the largest possible possitive balances in the dataset
high_balance = df[df["balance"] > 320039.575]

print("Balances above IQR upper bound:", len(high_balance))

display(
    high_balance[
        ["customer_id", "balance", "age", "credit_score"]
    ]
    .sort_values("balance", ascending=False)
    .head(20)
)

Balances above IQR upper bound: 200


,customer_id,balance,age,credit_score
45,15628972.0,999999.99,32,699.0
69,15632272.0,999999.99,42,792.0
84,15747677.0,999999.99,69,656.0
206,15565996.0,999999.99,44,653.0
273,15569222.0,999999.99,32,781.0
312,15718852.0,999999.99,56,794.0
440,15781678.0,999999.99,31,470.0
451,15685226.0,999999.99,29,712.0
471,15689955.0,999999.99,40,461.0
514,15728333.0,999999.99,43,521.0


In [38]:
# Examine negative balance values
negative_balance = df[df["balance"] < 0]

print("Number of negative balances:", len(negative_balance))
print("Minimum balance:", negative_balance["balance"].min())
print("Maximum negative balance:", negative_balance["balance"].max())

display(
    negative_balance["balance"]
    .value_counts()
    .head(20)
)

# Check whether negative balances contain repeated suspicious values
negative_counts = (
    negative_balance["balance"]
    .value_counts()
)

print("Number of unique negative balances:",
      negative_balance["balance"].nunique())

print("\nMost repeated negative balances:")
display(negative_counts.head(20))

Number of negative balances: 308
Minimum balance: -204510.94
Maximum negative balance: -40224.7


balance
-105473.74    2
-123805.03    1
-147249.29    1
-92242.34     1
-107818.63    1
-108166.09    1
-101486.96    1
-79731.91     1
-159024.71    1
-124577.15    1
-164113.04    1
-200724.96    1
-124513.66    1
-131287.36    1
-109219.83    1
-70417.79     1
-89409.95     1
-83036.05     1
-139887.01    1
-93796.42     1
Name: count, dtype: int64

Number of unique negative balances: 307

Most repeated negative balances:


balance
-105473.74    2
-123805.03    1
-147249.29    1
-92242.34     1
-107818.63    1
-108166.09    1
-101486.96    1
-79731.91     1
-159024.71    1
-124577.15    1
-164113.04    1
-200724.96    1
-124513.66    1
-131287.36    1
-109219.83    1
-70417.79     1
-89409.95     1
-83036.05     1
-139887.01    1
-93796.42     1
Name: count, dtype: int64

In [39]:
print(
    "999999.99 values:",
    (df["balance"] == 999999.99).sum()
)

print(
    df.loc[
        df["balance"] == 999999.99,
        ["customer_id", "balance", "age", "credit_score"]
    ].head(20)
)

# Replace injected noise with missing values
df["balance"] = df["balance"].replace(999999.99, np.nan)

print("Missing balances:", df["balance"].isna().sum())

999999.99 values: 200
      customer_id    balance  age  credit_score
45     15628972.0  999999.99   32         699.0
69     15632272.0  999999.99   42         792.0
84     15747677.0  999999.99   69         656.0
206    15565996.0  999999.99   44         653.0
273    15569222.0  999999.99   32         781.0
312    15718852.0  999999.99   56         794.0
440    15781678.0  999999.99   31         470.0
451    15685226.0  999999.99   29         712.0
471    15689955.0  999999.99   40         461.0
514    15728333.0  999999.99   43         521.0
541    15804787.0  999999.99   75         562.0
586    15796569.0  999999.99   44         831.0
597    15709928.0  999999.99   41         567.0
647    15763806.0  999999.99   41         773.0
769    15670080.0  999999.99   29         584.0
808    15722350.0  999999.99   37         627.0
878    15755225.0  999999.99   34         659.0
953    15611315.0  999999.99   23         708.0
957    15732307.0  999999.99   33         694.0
1021   15633574.0 

In [40]:
# calculate the median balance for imputation
balance_median = df["balance"].median()

print("Median balance:", balance_median)

# Impute missing balances with the median
df["balance"] = df["balance"].fillna(balance_median)

# verify that there are no more missing balances
print("Missing balances after cleaning:",
      df["balance"].isna().sum())

print("999999.99 values remaining:",
      (df["balance"] == 999999.99).sum())

print("Negative balances remaining:",
      (df["balance"] < 0).sum())

Median balance: 92566.53
Missing balances after cleaning: 0
999999.99 values remaining: 0
Negative balances remaining: 308


**Balance Outlier Handling**

The balance column contained two unusual patterns. First, 200 records had a balance of 999999.99, which appeared repeatedly as an identical extreme value. Second, 308 records had negative balances, with 307 unique negative values ranging from approximately -40,224.70 to -204,510.94.

The dataset documentation indicates that noise was intentionally injected into the data. Based on this information and the unusual patterns observed, these values were treated as injected noise rather than genuine account balances.

Because the original balance values could not be recovered from the available variables, the affected values were converted to missing values (NaN). The resulting missing balances were then imputed using the median of the valid balance values. Median imputation was selected because balance is a financial variable and can contain extreme values, making the median less sensitive to unusually large observations than the mean.

The remaining valid high balances were retained rather than automatically removed because an IQR outlier is not necessarily an erroneous value.

# Step 6 — Correct Data Types

In [41]:
df.dtypes

customer_id                float64
nome                           str
email                          str
numero_conta                   str
data_nascimento     datetime64[ns]
cidade                         str
country                        str
gender                         str
age                          int64
credit_score               float64
tenure                       int64
balance                    float64
limite_credito             float64
products_number              int64
credit_card                  int64
active_member                int64
estimated_salary           float64
churn                        int64
dtype: object

In [42]:
df.info()

<class 'pandas.DataFrame'>
Index: 10001 entries, 0 to 10148
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   customer_id       9798 non-null   float64       
 1   nome              10001 non-null  str           
 2   email             9601 non-null   str           
 3   numero_conta      10001 non-null  str           
 4   data_nascimento   10001 non-null  datetime64[ns]
 5   cidade            10001 non-null  str           
 6   country           10001 non-null  str           
 7   gender            10001 non-null  str           
 8   age               10001 non-null  int64         
 9   credit_score      10001 non-null  float64       
 10  tenure            10001 non-null  int64         
 11  balance           10001 non-null  float64       
 12  limite_credito    10001 non-null  float64       
 13  products_number   10001 non-null  int64         
 14  credit_card       10001 non-null  int6

In [43]:
# Convert customer_id from float to nullable integer, then to string
df["customer_id"] = df["customer_id"].astype("Int64").astype("str")

print("customer_id dtype:", df["customer_id"].dtype)
display(df["customer_id"].head(10))

customer_id dtype: str


0    15631460
1    15700300
2         NaN
3    15760456
4    15638730
5    15772482
6    15782236
7    15658987
8    15689061
9    15715622
Name: customer_id, dtype: str

In [44]:
print(
    "Credit scores with decimals:",
    (df["credit_score"] % 1 != 0).sum()
)

# Convert credit_score from float to nullable integer
df["credit_score"] = df["credit_score"].astype("int64")

print("Credit score dtype:", df["credit_score"].dtype)

Credit scores with decimals: 0
Credit score dtype: int64


In [45]:
df.dtypes

customer_id                    str
nome                           str
email                          str
numero_conta                   str
data_nascimento     datetime64[ns]
cidade                         str
country                        str
gender                         str
age                          int64
credit_score                 int64
tenure                       int64
balance                    float64
limite_credito             float64
products_number              int64
credit_card                  int64
active_member                int64
estimated_salary           float64
churn                        int64
dtype: object

In [46]:
# Final number of rows
print("Number of rows:", len(df))

# Final number of duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Final missing values by column
print("\nMissing values:")
display(df.isna().sum())

# Final data types
print("\nData types:")
display(df.dtypes)

Number of rows: 10001
Duplicate rows: 0

Missing values:


customer_id         203
nome                  0
email               400
numero_conta          0
data_nascimento       0
cidade                0
country               0
gender                0
age                   0
credit_score          0
tenure                0
balance               0
limite_credito        0
products_number       0
credit_card           0
active_member         0
estimated_salary      0
churn                 0
dtype: int64


Data types:


customer_id                    str
nome                           str
email                          str
numero_conta                   str
data_nascimento     datetime64[ns]
cidade                         str
country                        str
gender                         str
age                          int64
credit_score                 int64
tenure                       int64
balance                    float64
limite_credito             float64
products_number              int64
credit_card                  int64
active_member                int64
estimated_salary           float64
churn                        int64
dtype: object

**<h3>Step 7 : Before vs After Summary</h3>**

In [47]:
expected = {
    "customer_id": "str",
    "nome": "str",
    "email": "str",
    "data_nascimento": "datetime64[ns]",
    "age": "int64",
    "credit_score": "int64",
    "balance": "float64",
    "estimated_salary": "float64"
}


def placeholder_count(data):
    """Count placeholder values such as 'N/A' or 'unknown'."""
    placeholders = ["N/A", "n/a", "NA", "unknown", "Unknown", "None", ""]

    count = 0

    for column in data.select_dtypes(include=["object", "string"]).columns:
        count += data[column].astype("string").str.strip().isin(placeholders).sum()

    return int(count)


def dtype_accuracy(data):
    """Count columns whose data types match the expected types."""
    correct = 0

    for column, expected_type in expected.items():
        if column in data.columns and data[column].dtype == expected_type:
            correct += 1

    return correct

In [48]:
summary = pd.DataFrame({
    "Before": [
        len(data), int(data.isna().sum().sum()), int(data.duplicated().sum()),
        placeholder_count(data), f"{dtype_accuracy(data)}/{len(expected)}",
    ],
    "After": [
        len(df), int(df.isna().sum().sum()), int(df.duplicated().sum()),
        placeholder_count(df), f"{dtype_accuracy(df)}/{len(expected)}",
    ],
}, index=["Row count", "Null values", "Duplicate rows", "Placeholder / invalid values", "Columns with correct dtype"])

display(summary)

,Before,After
Row count,10150,10001
Null values,608,603
Duplicate rows,149,0
Placeholder / invalid values,0,0
Columns with correct dtype,6/8,8/8


In [51]:
# where the remaining nulls are
display(df.isna().sum()[df.isna().sum() > 0].to_frame("remaining nulls"))

,remaining nulls
customer_id,203
email,400


In [49]:
# Save the cleaned dataset
output_path = "bank_churn_cleaned.csv"

df.to_csv(output_path, index=False)

print(f"Cleaned dataset saved as: {output_path}")

Cleaned dataset saved as: bank_churn_cleaned.csv


In [50]:
# Verify that the saved dataset matches the cleaned DataFrame

df_check = pd.read_csv(
    "bank_churn_cleaned.csv",
    dtype={
        "customer_id": "str"
    },
    parse_dates=["data_nascimento"]
)

print(df_check.dtypes)

customer_id                    str
nome                           str
email                          str
numero_conta                   str
data_nascimento     datetime64[us]
cidade                         str
country                        str
gender                         str
age                          int64
credit_score                 int64
tenure                       int64
balance                    float64
limite_credito             float64
products_number              int64
credit_card                  int64
active_member                int64
estimated_salary           float64
churn                        int64
dtype: object


### Saving the Cleaned Dataset

After completing the data cleaning process, the cleaned dataset was saved as bank_churn_cleaned.csv. The original dataset was preserved, and the cleaned data was saved as a separate file to prevent accidental loss of the raw data.

The saved file was loaded again to verify that the dataset was successfully written and that the cleaned records were preserved.